In [1]:
import pandas as pd
import numpy as np
import time

In [3]:
def load_data(file_path):
    """Load the dataset and print basic information."""
    df = pd.read_csv(file_path)
    print(f"Dataset loaded with shape: {df.shape}")
    return df

In [5]:
def count_nulls(df, show_columns=True):
    """Count null values in the dataset."""
    null_counts = df.isnull().sum()
    total_nulls = null_counts.sum()

    if show_columns and total_nulls > 0:
        print("Null counts by column:")
        for col in df.columns:
            nulls = df[col].isnull().sum()
            if nulls > 0:
                print(f"  {col}: {nulls}")

    print(f"Total null values: {total_nulls}")
    return total_nulls

In [6]:
def fill_null_values(df):
    """Fill all null values in the dataset according to specified rules."""
    start_time = time.time()
    df_filled = df.copy()

    # Convert date to datetime for proper sorting
    df_filled['date'] = pd.to_datetime(df_filled['date'])

    # Identify relevant columns
    career_cols = [col for col in df_filled.columns if col.startswith('career_')]
    inns_cols = [col for col in df_filled.columns if col.startswith('inns_')]
    venue_cols = [col for col in df_filled.columns if col.startswith('venue_')]
    season_cols = [col for col in df_filled.columns if col.startswith('ssn_')]

    # Define last5 columns explicitly
    last5_cols = ['last5_avg', 'last5_eco', 'last5_sr', 'last5_wpm', 'last5_opm']
    available_last5_cols = [col for col in last5_cols if col in df_filled.columns]

    print(f"Processing {len(career_cols)} career stats, {len(inns_cols)} innings stats, {len(venue_cols)} venue stats columns, and {len(season_cols)} season stats columns")
    if available_last5_cols:
        print(f"Found {len(available_last5_cols)} 'last5' columns: {', '.join(available_last5_cols)}")

    # Create a reference mapping between venue and career columns
    venue_to_career = {}
    for venue_col in venue_cols:
        stat_type = venue_col.split('_')[1]  # Extract 'avg', 'eco', etc.
        career_col = f"career_{stat_type}"
        if career_col in career_cols:
            venue_to_career[venue_col] = career_col

    # Get all unique bowlers
    bowlers = df_filled['bowler_id'].unique()
    print(f"Processing {len(bowlers)} unique bowlers")

    # Track filled values
    total_filled = 0

    # Process each bowler
    for i, bowler_id in enumerate(bowlers):
        if i > 0 and i % 100 == 0:
            print(f"Processed {i}/{len(bowlers)} bowlers ({100*i/len(bowlers):.1f}%)")

        # Extract and sort bowler's matches
        bowler_mask = df_filled['bowler_id'] == bowler_id
        bowler_indices = df_filled.index[bowler_mask].tolist()

        if not bowler_indices:
            continue

        # Sort indices by date
        sorted_indices = df_filled.loc[bowler_indices].sort_values('date').index.tolist()

        # OPTIMIZATION: Pre-compute frequently accessed data
        matches_data = df_filled.loc[sorted_indices, ['date', 'league', 'season', 'venue']].copy()

        # Track seasons played by this bowler
        seasons_played = {}  # {league: [season1, season2, ...]}

        # Rule 1: Fill first match with zeros for all stats including last5 and season
        first_match_idx = sorted_indices[0]
        first_match_season = matches_data.loc[first_match_idx, 'season']
        first_match_league = matches_data.loc[first_match_idx, 'league']

        # Track the first season for this bowler
        if first_match_league not in seasons_played:
            seasons_played[first_match_league] = []
        seasons_played[first_match_league].append(first_match_season)

        for col in career_cols + inns_cols + venue_cols + available_last5_cols + season_cols:
            if pd.isna(df_filled.loc[first_match_idx, col]):
                df_filled.loc[first_match_idx, col] = 0.0
                total_filled += 1

        # OPTIMIZATION: Build lookup dictionaries for each player
        leagues_seen = set()
        league_seasons = {}  # {league: set(seasons)}
        venues_seen = set()

        leagues_seen.add(first_match_league)
        league_seasons[first_match_league] = {first_match_season}

        # Process remaining matches
        for i in range(1, len(sorted_indices)):
            curr_idx = sorted_indices[i]
            prev_indices = sorted_indices[:i]

            curr_match = matches_data.loc[curr_idx]
            curr_league = curr_match['league']
            curr_season = curr_match['season']
            curr_venue = curr_match['venue']

            # OPTIMIZATION: Use cached lookups
            first_in_league = curr_league not in leagues_seen
            first_in_season = not first_in_league and curr_season not in league_seasons.get(curr_league, set())
            first_at_venue = curr_venue not in venues_seen

            # Update tracking sets
            leagues_seen.add(curr_league)
            if curr_league not in league_seasons:
                league_seasons[curr_league] = set()
            league_seasons[curr_league].add(curr_season)
            venues_seen.add(curr_venue)

            # Track seasons for this player
            if curr_league not in seasons_played:
                seasons_played[curr_league] = []
            if curr_season not in seasons_played[curr_league]:
                seasons_played[curr_league].append(curr_season)

            # OPTIMIZATION: Create masks for filtered indices once
            if first_in_league:
                # No need to filter, use all previous indices
                prev_league_indices = []
            else:
                # Filter for same league
                prev_league_indices = [idx for idx in prev_indices
                                      if matches_data.loc[idx, 'league'] == curr_league]

            prev_at_venue_indices = [idx for idx in prev_indices
                                    if matches_data.loc[idx, 'venue'] == curr_venue]

            # Fill career stats
            for col in career_cols:
                if pd.isna(df_filled.loc[curr_idx, col]):
                    # Rule 2: First in league - use average from other leagues
                    if first_in_league:
                        prev_values = df_filled.loc[prev_indices, col].dropna()
                        if len(prev_values) > 0:
                            df_filled.loc[curr_idx, col] = prev_values.mean()
                        else:
                            df_filled.loc[curr_idx, col] = 0.0
                        total_filled += 1

                    # Rule 3: First in season - use average from same league
                    elif first_in_season:
                        prev_league_values = df_filled.loc[prev_league_indices, col].dropna()
                        if len(prev_league_values) > 0:
                            df_filled.loc[curr_idx, col] = prev_league_values.mean()
                        else:
                            prev_values = df_filled.loc[prev_indices, col].dropna()
                            df_filled.loc[curr_idx, col] = prev_values.mean() if len(prev_values) > 0 else 0.0
                        total_filled += 1

                    # Otherwise use most recent value (or the average of all previous)
                    else:
                        prev_values = df_filled.loc[prev_indices, col].dropna()
                        df_filled.loc[curr_idx, col] = prev_values.mean() if len(prev_values) > 0 else 0.0
                        total_filled += 1

            # Fill innings stats (use average of previous innings)
            for col in inns_cols:
                if pd.isna(df_filled.loc[curr_idx, col]):
                    prev_values = df_filled.loc[prev_indices, col].dropna()
                    df_filled.loc[curr_idx, col] = prev_values.mean() if len(prev_values) > 0 else 0.0
                    total_filled += 1

            # Fill venue stats
            for col in venue_cols:
                if pd.isna(df_filled.loc[curr_idx, col]):
                    # Rule 4: Venue debut - use average of previous venues
                    if first_at_venue:
                        # Get values for this same venue stat from all previous venues
                        prev_venue_values = df_filled.loc[prev_indices, col].dropna()
                        if len(prev_venue_values) > 0:
                            df_filled.loc[curr_idx, col] = prev_venue_values.mean()
                        else:
                            # OPTIMIZATION: Use pre-computed mapping
                            career_col = venue_to_career.get(col)
                            if career_col:
                                prev_career_values = df_filled.loc[prev_indices, career_col].dropna()
                                df_filled.loc[curr_idx, col] = prev_career_values.mean() if len(prev_career_values) > 0 else 0.0
                            else:
                                df_filled.loc[curr_idx, col] = 0.0
                        total_filled += 1
                    else:
                        # Not first at venue, use previous stats for this venue
                        if prev_at_venue_indices:
                            prev_venue_values = df_filled.loc[prev_at_venue_indices, col].dropna()
                            if len(prev_venue_values) > 0:
                                df_filled.loc[curr_idx, col] = prev_venue_values.mean()
                            else:
                                # Fallback to all previous venue stats
                                all_venue_values = df_filled.loc[prev_indices, col].dropna()
                                df_filled.loc[curr_idx, col] = all_venue_values.mean() if len(all_venue_values) > 0 else 0.0
                        else:
                            all_venue_values = df_filled.loc[prev_indices, col].dropna()
                            df_filled.loc[curr_idx, col] = all_venue_values.mean() if len(all_venue_values) > 0 else 0.0
                        total_filled += 1

            # Handle season stats with the new rules
            for col in season_cols:
                if pd.isna(df_filled.loc[curr_idx, col]):
                    # Get all leagues this player has played in before this match
                    player_leagues = list(seasons_played.keys())

                    # Rule 1: If it's player's first season ever, set to 0
                    total_seasons_played = sum(len(seasons) for seasons in seasons_played.values())
                    if total_seasons_played <= 1:  # Only the current season
                        df_filled.loc[curr_idx, col] = 0.0
                        total_filled += 1

                    # Rule 2: First season in a different league
                    elif first_in_league:
                        # Find the previous season stats from other leagues
                        prev_season_indices = []
                        for prev_idx in prev_indices:
                            prev_league = matches_data.loc[prev_idx, 'league']
                            prev_season = matches_data.loc[prev_idx, 'season']

                            # Check if from just the previous season in any other league
                            for league in player_leagues:
                                if league != curr_league and prev_league == league:
                                    # Get the most recent season in that league
                                    if prev_season == seasons_played[league][-1]:
                                        prev_season_indices.append(prev_idx)

                        if prev_season_indices:
                            prev_season_values = df_filled.loc[prev_season_indices, col].dropna()
                            df_filled.loc[curr_idx, col] = prev_season_values.mean() if len(prev_season_values) > 0 else 0.0
                        else:
                            df_filled.loc[curr_idx, col] = 0.0
                        total_filled += 1

                    # Rule 3: Different season in already played league
                    elif first_in_season:
                        # Find the previous season in the same league
                        if len(seasons_played[curr_league]) > 1:
                            prev_season = seasons_played[curr_league][-2]  # Get the just previous season

                            # Get indices for the previous season in the same league
                            prev_season_indices = [idx for idx in prev_indices
                                                if matches_data.loc[idx, 'league'] == curr_league and
                                                matches_data.loc[idx, 'season'] == prev_season]

                            if prev_season_indices:
                                prev_season_values = df_filled.loc[prev_season_indices, col].dropna()
                                df_filled.loc[curr_idx, col] = prev_season_values.mean() if len(prev_season_values) > 0 else 0.0
                            else:
                                df_filled.loc[curr_idx, col] = 0.0
                        else:
                            df_filled.loc[curr_idx, col] = 0.0
                        total_filled += 1

                    # Default case: use the current season's stats so far
                    else:
                        current_season_indices = [idx for idx in prev_indices
                                               if matches_data.loc[idx, 'league'] == curr_league and
                                               matches_data.loc[idx, 'season'] == curr_season]

                        if current_season_indices:
                            current_season_values = df_filled.loc[current_season_indices, col].dropna()
                            df_filled.loc[curr_idx, col] = current_season_values.mean() if len(current_season_values) > 0 else 0.0
                        else:
                            df_filled.loc[curr_idx, col] = 0.0
                        total_filled += 1

            # Fill all last5 columns with 0
            for col in available_last5_cols:
                if pd.isna(df_filled.loc[curr_idx, col]):
                    df_filled.loc[curr_idx, col] = 0.0
                    total_filled += 1

    print(f"Filled a total of {total_filled} null values")
    end_time = time.time()
    print(f"Processing completed in {end_time - start_time:.2f} seconds")

    return df_filled

In [8]:
file_path = '/content/all_league_bowl_features.csv'


 # Load the data
print("Loading cricket bowling dataset...")
df = load_data(file_path)

    # Check null values before processing
print("\nCounting null values before processing:")
initial_nulls = count_nulls(df)

    # Fill null values
print("\nFilling null values...")
filled_df = fill_null_values(df)

    # Check for any remaining nulls
print("\nCounting null values after processing:")
remaining_nulls = count_nulls(filled_df)

    # If there are still nulls, apply fallback
if remaining_nulls > 0:
  print("\nApplying fallback strategy to fill remaining nulls with 0...")
  filled_df = filled_df.fillna(0.0)

        # Final null check
  print("\nFinal null count check:")
  final_nulls = count_nulls(filled_df)
  if final_nulls == 0:
            print("All null values have been successfully filled!")
else:
    print("All null values have not been successfully filled!")

    # Save the processed data
output_path = file_path.replace('.csv', '_filled_ssn.csv')
filled_df.to_csv(output_path, index=False)
print(f"\nProcessed data saved to {output_path}")

Loading cricket bowling dataset...
Dataset loaded with shape: (161041, 35)

Counting null values before processing:
Null counts by column:
  career_avg: 8372
  career_eco: 8372
  career_sr: 8372
  career_wpm: 8372
  career_opm: 8372
  inns_avg: 15340
  inns_eco: 15340
  inns_sr: 15340
  inns_wpm: 15340
  inns_opm: 15340
  venue_avg: 45844
  venue_eco: 45844
  venue_sr: 45844
  venue_wpm: 45844
  venue_opm: 45844
  ssn_avg: 26110
  ssn_eco: 26110
  ssn_sr: 26110
  ssn_wpm: 26110
  ssn_opm: 26110
  last5_avg: 4691
  last5_eco: 4691
  last5_sr: 4691
  last5_wpm: 4691
  last5_opm: 4691
Total null values: 501785

Filling null values...
Processing 5 career stats, 5 innings stats, 5 venue stats columns, and 5 season stats columns
Found 5 'last5' columns: last5_avg, last5_eco, last5_sr, last5_wpm, last5_opm
Processing 4691 unique bowlers
Processed 100/4691 bowlers (2.1%)
Processed 200/4691 bowlers (4.3%)
Processed 300/4691 bowlers (6.4%)
Processed 400/4691 bowlers (8.5%)
Processed 500/4691 bow

In [10]:
df2=pd.read_csv('/content/all_league_bowl_features_filled_ssn.csv')


In [11]:
df2[(df2.bowler_id==625383) & (df2.league=='T20IS')].iloc[0]


,63850
date,2016-01-26
match_id,895817
bowl_team,India
bat_team,Australia
bowler_id,625383
bowler,JJ Bumrah
inns,2
venue,Adelaide Oval
season,2016
league,T20IS


In [12]:
# Add Career Debut Flag
#
# This script adds a 'flag' column to the cricket bowling dataset where:
# - 1 means career debut (first match ever played by a bowler)
# - 0 means all other matches
#
# Author: Avignyanno
# Date: 2025-04-10 23:16:02

# Import necessary libraries
import pandas as pd
import time

# File paths
input_path = '/content/all_league_bowl_features_filled_ssn.csv'
output_path = 'all_league_bowl_features_flagged.csv'

# Load the dataset
print(f"Loading data from {input_path}...")
df = pd.read_csv(input_path)
print(f"Dataset loaded with shape: {df.shape}")

# Ensure date is in datetime format for sorting
df['date'] = pd.to_datetime(df['date'])

# Initialize flag column with zeros
df['flag'] = 0

# Get unique bowlers
bowlers = df['bowler_id'].unique()
print(f"Processing {len(bowlers)} unique bowlers...")

# Start timing
start_time = time.time()

# Mark debuts for each bowler
debut_count = 0
for i, bowler_id in enumerate(bowlers):
    if i > 0 and i % 100 == 0:
        print(f"Processed {i}/{len(bowlers)} bowlers ({100*i/len(bowlers):.1f}%)")

    # Find all matches for this bowler
    bowler_matches = df[df['bowler_id'] == bowler_id]

    # Sort by date to find the first match
    first_match_idx = bowler_matches.sort_values('date').index[0]

    # Set flag to 1 for the first match
    df.at[first_match_idx, 'flag'] = 1
    debut_count += 1

# Calculate processing time
end_time = time.time()
processing_time = end_time - start_time

print(f"Marked {debut_count} career debuts with flag=1")
print(f"Processing completed in {processing_time:.2f} seconds")

# Save the dataset with the added flag column
df.to_csv(output_path, index=False)
print(f"Saved flagged dataset to {output_path}")
print(f"Final dataset shape: {df.shape}")

# Verify the results
total_ones = df['flag'].sum()
print(f"\nVerification:")
print(f"Total debut flags (should equal number of unique bowlers): {total_ones}")
print(f"Number of unique bowlers: {len(bowlers)}")

# Check a couple of random bowlers
sample_bowlers = bowlers[:3]  # Take first 3 bowlers as a sample
for bowler_id in sample_bowlers:
    bowler_matches = df[df['bowler_id'] == bowler_id].sort_values('date')
    first_match = bowler_matches.iloc[0]
    print(f"\nBowler ID: {bowler_id}")
    print(f"First match date: {first_match['date']}")
    print(f"Flag value: {first_match['flag']}")
    print(f"Total matches: {len(bowler_matches)}")

print("\nScript completed successfully!")

Loading data from /content/all_league_bowl_features_filled_ssn.csv...
Dataset loaded with shape: (161041, 35)
Processing 4691 unique bowlers...
Processed 100/4691 bowlers (2.1%)
Processed 200/4691 bowlers (4.3%)
Processed 300/4691 bowlers (6.4%)
Processed 400/4691 bowlers (8.5%)
Processed 500/4691 bowlers (10.7%)
Processed 600/4691 bowlers (12.8%)
Processed 700/4691 bowlers (14.9%)
Processed 800/4691 bowlers (17.1%)
Processed 900/4691 bowlers (19.2%)
Processed 1000/4691 bowlers (21.3%)
Processed 1100/4691 bowlers (23.4%)
Processed 1200/4691 bowlers (25.6%)
Processed 1300/4691 bowlers (27.7%)
Processed 1400/4691 bowlers (29.8%)
Processed 1500/4691 bowlers (32.0%)
Processed 1600/4691 bowlers (34.1%)
Processed 1700/4691 bowlers (36.2%)
Processed 1800/4691 bowlers (38.4%)
Processed 1900/4691 bowlers (40.5%)
Processed 2000/4691 bowlers (42.6%)
Processed 2100/4691 bowlers (44.8%)
Processed 2200/4691 bowlers (46.9%)
Processed 2300/4691 bowlers (49.0%)
Processed 2400/4691 bowlers (51.2%)
Proce

In [13]:
df3=pd.read_csv('/content/all_league_bowl_features_flagged.csv')


In [15]:
df3[(df3.bowler_id==625383) & (df3.league=='T20IS')].iloc[0]


,63850
date,2016-01-26
match_id,895817
bowl_team,India
bat_team,Australia
bowler_id,625383
bowler,JJ Bumrah
inns,2
venue,Adelaide Oval
season,2016
league,T20IS


In [16]:
df3[(df3.bowler_id==625383) & (df3.league=='IPL')].iloc[0]


,63824
date,2013-04-04
match_id,597999
bowl_team,Mumbai Indians
bat_team,Royal Challengers Bangalore
bowler_id,625383
bowler,JJ Bumrah
inns,1
venue,M.Chinnaswamy Stadium
season,2013
league,IPL
